# Experience Replay & Advanced DQN Tricks

## Module 6.5 — Deep RL: Prioritized Replay, Double DQN, Dueling DQN, and More

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_06_Deep_RL/05_Experience_Replay_And_Tricks/experience_replay_and_tricks.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:

1. **Implement** Prioritized Experience Replay (PER) with proportional prioritization
2. **Understand** n-step returns and their bias-variance trade-off
3. **Build** Double DQN to reduce overestimation bias
4. **Design** the Dueling DQN architecture with separate value and advantage streams
5. **Implement** Noisy Networks for exploration without epsilon-greedy
6. **Compare** each technique with before/after visualizations

In [ ]:
# Install dependencies (Colab-compatible)
!pip install torch torchvision numpy matplotlib pillow scipy --quiet

In [ ]:
# Download REAL open-source dataset for Deep RL image environment
import torchvision
import torchvision.transforms as transforms
import numpy as np

# CIFAR-10 as our image environment (real photos to enhance)
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
real_images = np.array([np.array(cifar10[i][0]) for i in range(500)])
print(f"✅ CIFAR-10 loaded: {len(real_images)} real photos as RL environment images")
print(f"   Shape: {real_images[0].shape} (32x32 RGB)")
print(f"   Agent will learn to enhance these REAL photographs!")

# Pre-trained model weights (for feature extraction)
from torchvision import models
print("✅ Pre-trained ResNet18 weights available for state encoding")

## Deep Derivation: Experience Replay, Prioritization, and Double DQN

### Step 1: Uniform Experience Replay — Breaking Correlation
Transitions $(s_t, a_t, r_t, s_{t+1})$ stored in buffer $\mathcal{D}$ of size $N$.

**Why correlated data hurts SGD:**

SGD convergence requires $E[\hat{g}] = \nabla L$ (unbiased) and bounded variance. Sequential transitions are correlated:
$$\text{Cov}(\hat{g}_t, \hat{g}_{t+1}) \neq 0$$

Uniform sampling from $\mathcal{D}$ breaks temporal correlation:
$$E_{(s,a,r,s') \sim \text{Uniform}(\mathcal{D})}[\hat{g}] = \nabla L \quad \blacksquare$$

### Step 2: Prioritized Experience Replay (PER)
Sample transition $i$ with probability:
$$P(i) = \frac{p_i^\alpha}{\sum_k p_k^\alpha}$$

where $p_i = |\delta_i| + \epsilon$ and $\delta_i$ is the TD error.

**Parameter $\alpha$:** controls prioritization strength ($\alpha=0$: uniform, $\alpha=1$: full priority).

### Step 3: Importance Sampling Correction for PER
PER introduces bias (non-uniform sampling). Correct with importance sampling weights:
$$w_i = \left(\frac{1}{N} \cdot \frac{1}{P(i)}\right)^\beta$$

**Proof this corrects the bias:**

For expectation under uniform distribution $U$:
$$E_U[f(x)] = \sum_i \frac{1}{N} f(x_i) = \sum_i P(i) \cdot \frac{1}{N \cdot P(i)} \cdot f(x_i) = E_P[w_i \cdot f(x_i)] \quad \blacksquare$$

As $\beta \to 1$: fully corrected. Schedule: $\beta$ anneals from $\beta_0$ to $1$ during training.

### Step 4: Double DQN — Correcting Maximization Bias
**Problem:** $\max_{a'} Q(s', a'; \theta)$ overestimates because $E[\max_i X_i] \geq \max_i E[X_i]$.

**Proof of maximization bias (Jensen's inequality):**

$\max$ is a convex function. By Jensen's inequality:
$$E[\max_{a'} Q(s', a')] \geq \max_{a'} E[Q(s', a')] = \max_{a'} Q^*(s', a') \quad \blacksquare$$

**Double DQN solution:** Decouple action selection from evaluation:
$$y_t = r_t + \gamma Q(s_{t+1}, \underbrace{\arg\max_{a'} Q(s_{t+1}, a'; \theta)}_{\text{select with } \theta}; \underbrace{\theta^-}_{\text{evaluate with } \theta^-})$$

### Step 5: Dueling DQN Architecture
Decompose Q-function:
$$Q(s, a; \theta, \alpha, \beta) = V(s; \theta, \beta) + \left(A(s, a; \theta, \alpha) - \frac{1}{|\mathcal{A}|}\sum_{a'} A(s, a'; \theta, \alpha)\right)$$

**Proof this decomposition identifies $V$ uniquely:**

Taking $\max_a$: $\max_a Q(s,a) = V(s) + \max_a A(s,a) - \text{mean}[A]$

Subtracting mean ensures $\sum_a A(s,a) = 0$, making $V$ identifiable. Without centering, $V$ and $A$ are not uniquely determined (can shift constants between them). $\blacksquare$

### Step 6: Target Network Update Strategies
**Hard update:** $\theta^- \leftarrow \theta$ every $C$ steps.

**Soft (Polyak) update:** $\theta^- \leftarrow \tau\theta + (1-\tau)\theta^-$

**Proof soft update is an exponential moving average:**

After $T$ updates: $\theta^-_T = \tau\sum_{k=0}^{T-1}(1-\tau)^k \theta_{T-k} + (1-\tau)^T \theta^-_0$

**Proof:** By induction. Base: $\theta^-_1 = \tau\theta_1 + (1-\tau)\theta^-_0$. Step: substitute recursively. $\blacksquare$

Effective window: $\sim 1/\tau$ steps (for $\tau=0.005$: averages over ~200 updates).

### Step 7: Gradient Clipping — Stable Training
Clip gradient norm: $\hat{g} = \min\left(1, \frac{G_{\max}}{\|\nabla_\theta L\|}\right) \nabla_\theta L$

**Connection to RL for vision:** These tricks (PER, Double DQN, Dueling) are essential when using deep CNNs as Q-networks for image-based tasks, where high-dimensional state spaces make training unstable without careful stabilization.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import deque, namedtuple
from scipy.ndimage import convolve
import random
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

---

## 1. Prioritized Experience Replay (PER)

### 1.1 Motivation

Uniform sampling from the replay buffer treats all transitions equally. But some transitions are more **informative** than others:

- Transitions with large TD error $|\delta_t|$ indicate the agent's prediction is significantly wrong
- These are the transitions we should learn from most

### 1.2 Proportional Prioritization

Assign priority to each transition based on TD error:

$$p_i = |\delta_i| + \epsilon_{\text{per}}$$

where $\epsilon_{\text{per}}$ is a small constant to ensure all transitions have non-zero probability.

Sampling probability:

$$P(i) = \frac{p_i^\alpha}{\sum_k p_k^\alpha}$$

where $\alpha \in [0, 1]$ controls prioritization:
- $\alpha = 0$: uniform sampling
- $\alpha = 1$: full prioritization

### 1.3 Importance Sampling Correction

Prioritized sampling introduces bias. We correct with importance sampling weights:

$$w_i = \left(\frac{1}{N} \cdot \frac{1}{P(i)}\right)^\beta$$

Normalized: $\hat{w}_i = \frac{w_i}{\max_j w_j}$

The $\beta$ parameter is annealed from $\beta_0$ to $1$ during training:

$$\beta_t = \beta_0 + (1 - \beta_0) \cdot \frac{t}{T}$$

### 1.4 Sum Tree Data Structure

Efficient $O(\log N)$ sampling using a sum tree — a binary tree where each parent is the sum of its children.

In [ ]:
class SumTree:
    """Binary sum tree for O(log n) proportional sampling."""

    def __init__(self, capacity):
        self.capacity = capacity
        self.tree = np.zeros(2 * capacity - 1)
        self.data = np.zeros(capacity, dtype=object)
        self.size = 0
        self.write_idx = 0

    def _propagate(self, idx, change):
        parent = (idx - 1) // 2
        self.tree[parent] += change
        if parent != 0:
            self._propagate(parent, change)

    def _retrieve(self, idx, s):
        left = 2 * idx + 1
        right = left + 1
        if left >= len(self.tree):
            return idx
        if s <= self.tree[left]:
            return self._retrieve(left, s)
        else:
            return self._retrieve(right, s - self.tree[left])

    @property
    def total(self):
        return self.tree[0]

    def add(self, priority, data):
        idx = self.write_idx + self.capacity - 1
        self.data[self.write_idx] = data
        self.update(idx, priority)
        self.write_idx = (self.write_idx + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def update(self, idx, priority):
        change = priority - self.tree[idx]
        self.tree[idx] = priority
        self._propagate(idx, change)

    def get(self, s):
        idx = self._retrieve(0, s)
        data_idx = idx - self.capacity + 1
        return idx, self.tree[idx], self.data[data_idx]


class PrioritizedReplayBuffer:
    """Prioritized Experience Replay with sum tree."""

    def __init__(self, capacity, alpha=0.6, beta_start=0.4, beta_frames=100000):
        self.tree = SumTree(capacity)
        self.capacity = capacity
        self.alpha = alpha
        self.beta_start = beta_start
        self.beta_frames = beta_frames
        self.frame = 0
        self.max_priority = 1.0
        self.epsilon_per = 1e-6

    @property
    def beta(self):
        return min(1.0, self.beta_start + self.frame * (1.0 - self.beta_start) / self.beta_frames)

    def push(self, state, action, reward, next_state, done):
        priority = self.max_priority ** self.alpha
        self.tree.add(priority, (state, action, reward, next_state, done))

    def sample(self, batch_size):
        self.frame += 1
        batch = []
        indices = []
        priorities = []
        segment = self.tree.total / batch_size

        for i in range(batch_size):
            s = random.uniform(segment * i, segment * (i + 1))
            idx, priority, data = self.tree.get(s)
            if data == 0:  # Empty slot
                continue
            batch.append(data)
            indices.append(idx)
            priorities.append(priority)

        # Importance sampling weights
        N = self.tree.size
        probs = np.array(priorities) / self.tree.total
        weights = (N * probs) ** (-self.beta)
        weights /= weights.max()

        states = np.array([b[0] for b in batch])
        actions = np.array([b[1] for b in batch])
        rewards = np.array([b[2] for b in batch])
        next_states = np.array([b[3] for b in batch])
        dones = np.array([b[4] for b in batch])

        return states, actions, rewards, next_states, dones, indices, weights

    def update_priorities(self, indices, td_errors):
        for idx, td_error in zip(indices, td_errors):
            priority = (abs(td_error) + self.epsilon_per) ** self.alpha
            self.tree.update(idx, priority)
            self.max_priority = max(self.max_priority, abs(td_error) + self.epsilon_per)

    def __len__(self):
        return self.tree.size


# Demonstrate PER vs Uniform sampling
print("PrioritizedReplayBuffer implemented.")
print(f"\nKey features:")
print(f"  - Sum tree for O(log N) sampling")
print(f"  - Importance sampling weights to correct bias")
print(f"  - Beta annealing from {0.4} to 1.0")

In [ ]:
# Visualize priority distribution

np.random.seed(42)

# Simulate TD errors from training
n_transitions = 5000
td_errors = np.abs(np.random.exponential(0.5, n_transitions))  # Most are small, few are large

# Proportional priorities
alpha = 0.6
priorities = (td_errors + 1e-6) ** alpha
probs_prioritized = priorities / priorities.sum()
probs_uniform = np.ones(n_transitions) / n_transitions

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# TD error distribution
ax = axes[0]
ax.hist(td_errors, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax.set_xlabel('|TD Error|')
ax.set_ylabel('Count')
ax.set_title('Distribution of TD Errors', fontsize=11)
ax.grid(True, alpha=0.3)

# Sampling probability comparison
ax = axes[1]
sorted_indices = np.argsort(td_errors)
ax.plot(probs_uniform[sorted_indices] * n_transitions, label='Uniform', linewidth=2)
ax.plot(probs_prioritized[sorted_indices] * n_transitions, label=f'Prioritized ($\\alpha={alpha}$)', linewidth=2)
ax.set_xlabel('Transitions (sorted by |TD error|)')
ax.set_ylabel('Relative Sampling Probability')
ax.set_title('Sampling Probability: Uniform vs PER', fontsize=11)
ax.legend()
ax.grid(True, alpha=0.3)

# Beta annealing
ax = axes[2]
frames = np.arange(100000)
beta_values = np.minimum(1.0, 0.4 + frames * 0.6 / 100000)
ax.plot(frames, beta_values, color='purple', linewidth=2)
ax.set_xlabel('Training Frames')
ax.set_ylabel('$\\beta$')
ax.set_title('IS Weight Correction ($\\beta$ Annealing)', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.35, 1.05)

plt.suptitle('Prioritized Experience Replay', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 2. N-Step Returns

### 2.1 Definition

Instead of using 1-step TD target $r_t + \gamma V(s_{t+1})$, use n-step returns:

$$G_t^{(n)} = \sum_{k=0}^{n-1} \gamma^k r_{t+k} + \gamma^n V(s_{t+n})$$

For DQN, the n-step target becomes:

$$y_t^{(n)} = \sum_{k=0}^{n-1} \gamma^k r_{t+k} + \gamma^n \max_{a'} Q(s_{t+n}, a'; \theta^-)$$

### 2.2 Bias-Variance Analysis

$$\text{Var}[G_t^{(n)}] = \sum_{k=0}^{n-1} \gamma^{2k} \text{Var}[r_{t+k}] + \gamma^{2n} \text{Var}[V(s_{t+n})]$$

- **Small $n$**: Low variance, high bias (relies on potentially inaccurate $V$)
- **Large $n$**: High variance, low bias (uses more real rewards)
- **Optimal $n$**: Typically 3–5 for most tasks

### 2.3 Implementation

Store n-step transitions $(s_t, a_t, G_t^{(n)}, s_{t+n})$ in the replay buffer.

In [ ]:
class NStepBuffer:
    """N-step return buffer for computing multi-step targets."""

    def __init__(self, n_steps=3, gamma=0.99):
        self.n_steps = n_steps
        self.gamma = gamma
        self.buffer = deque(maxlen=n_steps)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def get_nstep_transition(self):
        """Compute n-step return and return the transition."""
        if len(self.buffer) < self.n_steps:
            return None

        # Compute n-step return
        R = 0
        for i in range(self.n_steps):
            _, _, r, _, d = self.buffer[i]
            R += (self.gamma ** i) * r
            if d:
                # Truncate at terminal state
                n_actual = i + 1
                _, _, _, next_s, _ = self.buffer[i]
                return self.buffer[0][0], self.buffer[0][1], R, next_s, True, n_actual

        # No terminal state in n steps
        _, _, _, next_s, _ = self.buffer[-1]
        return self.buffer[0][0], self.buffer[0][1], R, next_s, False, self.n_steps

    def reset(self):
        self.buffer.clear()


# Demonstrate n-step returns
print("N-Step Return Example:")
print("="*50)

gamma = 0.99
rewards = [1.0, 0.5, 2.0, -0.5, 1.0]

for n in [1, 2, 3, 5]:
    G_n = sum(gamma**k * rewards[k] for k in range(min(n, len(rewards))))
    print(f"  n={n}: G_t^({n}) = {G_n:.4f} + γ^{n} * V(s_{{t+{n}}})")

print(f"\n  Full MC return: G_t = {sum(gamma**k * r for k, r in enumerate(rewards)):.4f}")

---

## 3. Double DQN

### 3.1 The Overestimation Problem

Standard DQN uses the same network to **select** and **evaluate** actions:

$$y_t = r + \gamma \max_{a'} Q(s', a'; \theta^-)$$

The $\max$ operator introduces a **positive bias**:

$$\mathbb{E}\left[\max_{a'} Q(s', a'; \theta^-)\right] \geq \max_{a'} \mathbb{E}\left[Q(s', a'; \theta^-)\right]$$

This is Jensen's inequality applied to the convex $\max$ function. The noise in Q-value estimates gets amplified by the max operation.

### 3.2 Double DQN Solution

**Decouple selection from evaluation** (van Hasselt et al., 2016):

1. **Select** best action using the online network $\theta$:
   $$a^* = \arg\max_{a'} Q(s', a'; \theta)$$

2. **Evaluate** that action using the target network $\theta^-$:
   $$y_t = r + \gamma Q(s', a^*; \theta^-)$$

Combined:
$$y_t^{\text{DDQN}} = r + \gamma Q\left(s', \arg\max_{a'} Q(s', a'; \theta); \; \theta^-\right)$$

### 3.3 Why This Reduces Overestimation

If the online network overestimates action $a'$, it will select it, BUT the target network (which has different noise) is unlikely to also overestimate the same action. The overestimation errors are decorrelated.

In [ ]:
# Demonstrate overestimation bias

np.random.seed(42)

n_actions = 10
true_q = np.zeros(n_actions)  # True Q-values are all 0
n_experiments = 2000
noise_levels = [0.5, 1.0, 2.0, 3.0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overestimation vs noise level
dqn_overest = []
ddqn_overest = []

for noise in noise_levels:
    dqn_vals = []
    ddqn_vals = []

    for _ in range(n_experiments):
        # DQN: max of noisy estimates
        noisy_q = true_q + np.random.normal(0, noise, n_actions)
        dqn_vals.append(np.max(noisy_q))

        # Double DQN: select with one, evaluate with another
        noisy_q1 = true_q + np.random.normal(0, noise, n_actions)  # Online net
        noisy_q2 = true_q + np.random.normal(0, noise, n_actions)  # Target net
        best_action = np.argmax(noisy_q1)
        ddqn_vals.append(noisy_q2[best_action])

    dqn_overest.append(np.mean(dqn_vals))
    ddqn_overest.append(np.mean(ddqn_vals))

ax = axes[0]
x = np.arange(len(noise_levels))
width = 0.35
ax.bar(x - width/2, dqn_overest, width, label='DQN (overestimates)', color='red', alpha=0.7)
ax.bar(x + width/2, ddqn_overest, width, label='Double DQN (corrected)', color='blue', alpha=0.7)
ax.axhline(y=0, color='black', linestyle='--', alpha=0.5, label='True Value = 0')
ax.set_xticks(x)
ax.set_xticklabels([f'$\\sigma={n}$' for n in noise_levels])
ax.set_ylabel('Estimated Max Q-value')
ax.set_title('Overestimation Bias vs Noise Level', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

# Distribution comparison
ax = axes[1]
noise = 2.0
dqn_samples = [np.max(true_q + np.random.normal(0, noise, n_actions)) for _ in range(2000)]
ddqn_samples = []
for _ in range(2000):
    q1 = true_q + np.random.normal(0, noise, n_actions)
    q2 = true_q + np.random.normal(0, noise, n_actions)
    ddqn_samples.append(q2[np.argmax(q1)])

ax.hist(dqn_samples, bins=40, alpha=0.6, color='red', label=f'DQN (mean={np.mean(dqn_samples):.2f})', density=True)
ax.hist(ddqn_samples, bins=40, alpha=0.6, color='blue', label=f'DDQN (mean={np.mean(ddqn_samples):.2f})', density=True)
ax.axvline(x=0, color='black', linestyle='--', linewidth=2, label='True Value')
ax.set_xlabel('Estimated Value')
ax.set_ylabel('Density')
ax.set_title(f'Value Estimate Distribution ($\\sigma={noise}$, {n_actions} actions)', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Double DQN: Reducing Overestimation Bias', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 4. Dueling DQN Architecture

### 4.1 Key Insight

The Q-value can be decomposed:

$$Q(s, a) = V(s) + A(s, a)$$

where:
- $V(s)$: How good is the state (regardless of action)
- $A(s, a)$: How much better is action $a$ compared to the average

**Constraint**: $\sum_a \pi(a|s) A(s,a) = 0$ (advantages sum to zero under policy)

### 4.2 Dueling Architecture

Instead of directly outputting $Q(s,a)$, split the network into two streams:

$$\text{CNN features} \begin{cases} \rightarrow V(s; \theta, \alpha) & \text{(value stream)} \\ \rightarrow A(s, a; \theta, \beta) & \text{(advantage stream)} \end{cases}$$

### 4.3 Aggregation

To ensure identifiability (uniquely decompose $Q$ into $V$ and $A$):

$$Q(s, a; \theta, \alpha, \beta) = V(s; \theta, \alpha) + \left(A(s, a; \theta, \beta) - \frac{1}{|\mathcal{A}|} \sum_{a'} A(s, a'; \theta, \beta)\right)$$

Subtracting the mean ensures the advantage stream has zero mean.

### 4.4 Benefits

- **Better generalization**: The value stream learns state quality even without trying every action
- **Faster learning**: In many states, the action choice doesn't matter much (e.g., safe states)
- **Particularly useful** when many actions have similar values

In [ ]:
class DuelingDQN(nn.Module):
    """Dueling DQN with separate value and advantage streams."""

    def __init__(self, n_actions, image_size=48):
        super(DuelingDQN, self).__init__()

        # Shared feature extractor
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(4)
        )

        feature_size = 64 * 4 * 4

        # Value stream: V(s)
        self.value_stream = nn.Sequential(
            nn.Linear(feature_size, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

        # Advantage stream: A(s, a)
        self.advantage_stream = nn.Sequential(
            nn.Linear(feature_size, 256),
            nn.ReLU(),
            nn.Linear(256, n_actions)
        )

    def forward(self, x):
        features = self.features(x)
        features = features.view(features.size(0), -1)

        value = self.value_stream(features)             # [B, 1]
        advantages = self.advantage_stream(features)    # [B, n_actions]

        # Q = V + (A - mean(A))
        q_values = value + (advantages - advantages.mean(dim=1, keepdim=True))
        return q_values

    def get_value_and_advantages(self, x):
        """Return decomposed V and A for visualization."""
        features = self.features(x)
        features = features.view(features.size(0), -1)
        value = self.value_stream(features)
        advantages = self.advantage_stream(features)
        return value, advantages


# Compare architectures
class StandardDQN(nn.Module):
    """Standard DQN for comparison."""

    def __init__(self, n_actions, image_size=48):
        super(StandardDQN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(4)
        )
        self.q_head = nn.Sequential(
            nn.Linear(64 * 4 * 4, 512),
            nn.ReLU(),
            nn.Linear(512, n_actions)
        )

    def forward(self, x):
        features = self.features(x).view(x.size(0), -1)
        return self.q_head(features)


n_actions = 11
dueling = DuelingDQN(n_actions).to(device)
standard = StandardDQN(n_actions).to(device)

test_input = torch.randn(1, 3, 48, 48).to(device)
q_dueling = dueling(test_input)
q_standard = standard(test_input)
v, a = dueling.get_value_and_advantages(test_input)

print(f"Standard DQN params: {sum(p.numel() for p in standard.parameters()):,}")
print(f"Dueling DQN params: {sum(p.numel() for p in dueling.parameters()):,}")
print(f"\nDueling decomposition:")
print(f"  V(s) = {v.item():.4f}")
print(f"  A(s,a) = {a.detach().cpu().numpy().round(3)}")
print(f"  Q(s,a) = {q_dueling.detach().cpu().numpy().round(3)}")

In [ ]:
# Visualize Dueling Architecture conceptually

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Generate some test states
n_test = 50
test_states = torch.randn(n_test, 3, 48, 48).to(device)

with torch.no_grad():
    values, advantages = dueling.get_value_and_advantages(test_states)
    q_values = dueling(test_states)

values_np = values.cpu().numpy().flatten()
advantages_np = advantages.cpu().numpy()
q_np = q_values.cpu().numpy()

# Value distribution
ax = axes[0]
ax.hist(values_np, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
ax.set_xlabel('V(s)')
ax.set_ylabel('Count')
ax.set_title('State Values V(s)', fontsize=12)
ax.grid(True, alpha=0.3)

# Advantage distribution (per action)
ax = axes[1]
ax.boxplot(advantages_np, labels=[f'a{i}' for i in range(n_actions)], vert=True)
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Action')
ax.set_ylabel('A(s, a)')
ax.set_title('Advantage Values per Action', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

# Q-value decomposition for one state
ax = axes[2]
idx = 0
x_pos = np.arange(n_actions)
v_val = values_np[idx]
a_vals = advantages_np[idx] - advantages_np[idx].mean()
q_vals = q_np[idx]

ax.bar(x_pos, [v_val]*n_actions, alpha=0.4, color='blue', label=f'V(s)={v_val:.2f}')
ax.bar(x_pos, a_vals, bottom=v_val, alpha=0.6, color='orange', label='A(s,a) - mean(A)')
ax.scatter(x_pos, q_vals, color='red', zorder=5, s=50, label='Q(s,a) = V + A')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'a{i}' for i in range(n_actions)], fontsize=8)
ax.set_ylabel('Value')
ax.set_title('Q-Value Decomposition (One State)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('Dueling DQN Architecture Analysis', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 5. Noisy Networks

### 5.1 Motivation

Epsilon-greedy exploration is **state-independent** — it explores uniformly regardless of the state. Noisy Networks (Fortunato et al., 2018) add **learnable noise** to network weights:

$$y = (\mu^w + \sigma^w \odot \varepsilon^w) x + \mu^b + \sigma^b \odot \varepsilon^b$$

where $\varepsilon^w, \varepsilon^b$ are noise variables and $\mu, \sigma$ are learnable parameters.

### 5.2 Factorized Gaussian Noise

For efficiency, use factorized noise:

$$\varepsilon_{ij} = f(\varepsilon_i) \cdot f(\varepsilon_j)$$

where $f(x) = \text{sign}(x) \sqrt{|x|}$

This reduces noise parameters from $O(pq)$ to $O(p + q)$ for a $p \times q$ weight matrix.

### 5.3 Advantages

- **State-dependent exploration**: The network learns where to explore
- **No hyperparameter**: Eliminates $\epsilon$ scheduling
- **Self-annealing**: Noise naturally decreases as the agent becomes more certain

In [ ]:
class NoisyLinear(nn.Module):
    """Noisy linear layer with factorized Gaussian noise."""

    def __init__(self, in_features, out_features, sigma_init=0.5):
        super(NoisyLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.sigma_init = sigma_init

        # Learnable parameters
        self.weight_mu = nn.Parameter(torch.empty(out_features, in_features))
        self.weight_sigma = nn.Parameter(torch.empty(out_features, in_features))
        self.bias_mu = nn.Parameter(torch.empty(out_features))
        self.bias_sigma = nn.Parameter(torch.empty(out_features))

        # Factorized noise buffers
        self.register_buffer('weight_epsilon', torch.empty(out_features, in_features))
        self.register_buffer('bias_epsilon', torch.empty(out_features))

        self._init_parameters()
        self.reset_noise()

    def _init_parameters(self):
        mu_range = 1 / np.sqrt(self.in_features)
        self.weight_mu.data.uniform_(-mu_range, mu_range)
        self.weight_sigma.data.fill_(self.sigma_init / np.sqrt(self.in_features))
        self.bias_mu.data.uniform_(-mu_range, mu_range)
        self.bias_sigma.data.fill_(self.sigma_init / np.sqrt(self.out_features))

    @staticmethod
    def _factorized_noise(size):
        x = torch.randn(size)
        return x.sign() * x.abs().sqrt()

    def reset_noise(self):
        epsilon_in = self._factorized_noise(self.in_features)
        epsilon_out = self._factorized_noise(self.out_features)
        self.weight_epsilon.copy_(epsilon_out.outer(epsilon_in))
        self.bias_epsilon.copy_(epsilon_out)

    def forward(self, x):
        if self.training:
            weight = self.weight_mu + self.weight_sigma * self.weight_epsilon
            bias = self.bias_mu + self.bias_sigma * self.bias_epsilon
        else:
            weight = self.weight_mu
            bias = self.bias_mu
        return F.linear(x, weight, bias)


class NoisyDQN(nn.Module):
    """DQN with NoisyNet layers for exploration."""

    def __init__(self, n_actions, image_size=48):
        super(NoisyDQN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(4)
        )

        feature_size = 64 * 4 * 4

        # Noisy layers (no epsilon-greedy needed!)
        self.noisy_fc1 = NoisyLinear(feature_size, 256)
        self.noisy_fc2 = NoisyLinear(256, n_actions)

    def forward(self, x):
        features = self.features(x).view(x.size(0), -1)
        x = F.relu(self.noisy_fc1(features))
        return self.noisy_fc2(x)

    def reset_noise(self):
        self.noisy_fc1.reset_noise()
        self.noisy_fc2.reset_noise()


# Demonstrate noise behavior
noisy_net = NoisyDQN(n_actions=11).to(device)
test_state = torch.randn(1, 3, 48, 48).to(device)

# Multiple forward passes with different noise
noisy_net.train()
q_samples = []
for _ in range(100):
    noisy_net.reset_noise()
    with torch.no_grad():
        q_samples.append(noisy_net(test_state).cpu().numpy().flatten())

q_samples = np.array(q_samples)

print("NoisyNet Q-value statistics for one state:")
print(f"  Mean: {q_samples.mean(axis=0).round(3)}")
print(f"  Std:  {q_samples.std(axis=0).round(3)}")
print(f"  → Exploration is automatic through weight noise!")

In [ ]:
# Visualize NoisyNet exploration behavior

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Q-value distribution from noise
ax = axes[0]
for i in range(min(5, q_samples.shape[1])):
    ax.hist(q_samples[:, i], bins=20, alpha=0.5, label=f'Action {i}')
ax.set_xlabel('Q-value')
ax.set_ylabel('Count')
ax.set_title('NoisyNet: Q-value Distribution per Action\n(Same state, different noise)', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Sigma values (noise magnitude) during hypothetical training
ax = axes[1]
sigma_w = noisy_net.noisy_fc1.weight_sigma.detach().cpu().numpy().flatten()
sigma_b = noisy_net.noisy_fc1.bias_sigma.detach().cpu().numpy().flatten()
ax.hist(sigma_w, bins=30, alpha=0.7, color='blue', label='Weight $\\sigma$', density=True)
ax.hist(sigma_b, bins=15, alpha=0.7, color='orange', label='Bias $\\sigma$', density=True)
ax.set_xlabel('$\\sigma$ value')
ax.set_ylabel('Density')
ax.set_title('Learnable Noise Magnitudes ($\\sigma$ parameters)', fontsize=11)
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Noisy Networks: Learned Exploration', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 6. Combined Agent: Rainbow-Lite

Let's combine all tricks into a single agent and compare against vanilla DQN.

In [ ]:
class ImageEnvTricks:
    """Image processing environment for comparing DQN tricks."""

    ACTIONS = [
        'sharpen', 'denoise', 'brighten', 'darken',
        'contrast_up', 'contrast_down', 'gamma_up', 'gamma_down', 'no_op'
    ]

    def __init__(self, image_size=48, max_steps=8):
        self.image_size = image_size
        self.max_steps = max_steps
        self.n_actions = len(self.ACTIONS)
        self.step_count = 0

    def _generate_target(self):
        img = np.zeros((3, self.image_size, self.image_size), dtype=np.float32)
        x = np.linspace(0, 4*np.pi, self.image_size)
        y = np.linspace(0, 4*np.pi, self.image_size)
        X, Y = np.meshgrid(x, y)
        p = np.random.uniform(0, 2*np.pi)
        img[0] = 0.5 + 0.35 * np.sin(X + p) * np.cos(Y)
        img[1] = 0.5 + 0.30 * np.cos(X + Y + p)
        img[2] = 0.5 + 0.25 * np.sin(X - Y + p)
        return np.clip(img, 0, 1)

    def _degrade(self, img):
        d = img.copy()
        d += np.random.normal(0, 0.12, img.shape).astype(np.float32)
        d = 0.5 + 0.6 * (d - 0.5)
        d += np.random.uniform(-0.08, 0.08)
        return np.clip(d, 0, 1)

    def _psnr(self, a, b):
        mse = np.mean((a - b)**2)
        return 10 * np.log10(1.0 / max(mse, 1e-10))

    def reset(self):
        self.step_count = 0
        self.target = self._generate_target()
        self.current = self._degrade(self.target)
        self.prev_psnr = self._psnr(self.current, self.target)
        return self.current.copy()

    def step(self, action):
        self.step_count += 1
        img = self.current.copy()
        name = self.ACTIONS[action]

        if name == 'sharpen':
            k = np.array([[0, -0.3, 0], [-0.3, 2.2, -0.3], [0, -0.3, 0]])
            for c in range(3): img[c] = convolve(img[c], k, mode='reflect')
        elif name == 'denoise':
            k = np.ones((3, 3)) / 9.0
            for c in range(3): img[c] = convolve(img[c], k, mode='reflect')
        elif name == 'brighten':
            img += 0.04
        elif name == 'darken':
            img -= 0.04
        elif name == 'contrast_up':
            img = 0.5 + 1.2 * (img - 0.5)
        elif name == 'contrast_down':
            img = 0.5 + 0.85 * (img - 0.5)
        elif name == 'gamma_up':
            img = np.power(np.clip(img, 0, 1), 0.9)
        elif name == 'gamma_down':
            img = np.power(np.clip(img, 0, 1), 1.1)

        self.current = np.clip(img, 0, 1)
        new_psnr = self._psnr(self.current, self.target)
        reward = new_psnr - self.prev_psnr
        self.prev_psnr = new_psnr
        done = self.step_count >= self.max_steps
        return self.current.copy(), reward, done, {'psnr': new_psnr}


print(f"Environment ready: {len(ImageEnvTricks.ACTIONS)} actions")

In [ ]:
class EnhancedDQNAgent:
    """DQN agent combining: Double DQN + Dueling + PER + N-step."""

    def __init__(self, n_actions, image_size=48, lr=3e-4, gamma=0.95,
                 n_steps=3, use_double=True, use_per=True, use_dueling=True,
                 buffer_size=30000, batch_size=64, target_update=50):
        self.n_actions = n_actions
        self.gamma = gamma
        self.n_steps = n_steps
        self.use_double = use_double
        self.use_per = use_per
        self.batch_size = batch_size
        self.target_update = target_update
        self.steps_done = 0

        # Network
        if use_dueling:
            self.policy_net = DuelingDQN(n_actions, image_size).to(device)
            self.target_net = DuelingDQN(n_actions, image_size).to(device)
        else:
            self.policy_net = StandardDQN(n_actions, image_size).to(device)
            self.target_net = StandardDQN(n_actions, image_size).to(device)

        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)

        # Replay buffer
        if use_per:
            self.memory = PrioritizedReplayBuffer(buffer_size, alpha=0.6, beta_start=0.4)
        else:
            self.memory = deque(maxlen=buffer_size)

        # N-step buffer
        self.n_step_buffer = NStepBuffer(n_steps=n_steps, gamma=gamma)

        # Epsilon
        self.eps_start, self.eps_end, self.eps_decay = 1.0, 0.05, 2000
        self.losses = []

    def get_epsilon(self):
        return self.eps_end + (self.eps_start - self.eps_end) * np.exp(-self.steps_done / self.eps_decay)

    def select_action(self, state):
        self.steps_done += 1
        if random.random() < self.get_epsilon():
            return random.randrange(self.n_actions)
        with torch.no_grad():
            state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            return self.policy_net(state_t).argmax(1).item()

    def store_transition(self, state, action, reward, next_state, done):
        # N-step processing
        self.n_step_buffer.push(state, action, reward, next_state, done)
        result = self.n_step_buffer.get_nstep_transition()

        if result is not None:
            s, a, r_n, ns, d, n = result
            if self.use_per:
                self.memory.push(s, a, r_n, ns, float(d))
            else:
                self.memory.append((s, a, r_n, ns, float(d)))

        if done:
            self.n_step_buffer.reset()

    def optimize(self):
        min_size = self.batch_size
        if (self.use_per and len(self.memory) < min_size) or \
           (not self.use_per and len(self.memory) < min_size):
            return

        if self.use_per:
            states, actions, rewards, next_states, dones, indices, weights = \
                self.memory.sample(self.batch_size)
            weights_t = torch.FloatTensor(weights).to(device)
        else:
            batch = random.sample(self.memory, self.batch_size)
            states = np.array([b[0] for b in batch])
            actions = np.array([b[1] for b in batch])
            rewards = np.array([b[2] for b in batch])
            next_states = np.array([b[3] for b in batch])
            dones = np.array([b[4] for b in batch])
            weights_t = torch.ones(self.batch_size).to(device)

        states_t = torch.FloatTensor(states).to(device)
        actions_t = torch.LongTensor(actions).to(device)
        rewards_t = torch.FloatTensor(rewards).to(device)
        next_states_t = torch.FloatTensor(next_states).to(device)
        dones_t = torch.FloatTensor(dones).to(device)

        # Current Q
        current_q = self.policy_net(states_t).gather(1, actions_t.unsqueeze(1)).squeeze(1)

        # Target Q
        with torch.no_grad():
            if self.use_double:
                # Double DQN: select with online, evaluate with target
                next_actions = self.policy_net(next_states_t).argmax(1)
                next_q = self.target_net(next_states_t).gather(1, next_actions.unsqueeze(1)).squeeze(1)
            else:
                next_q = self.target_net(next_states_t).max(1)[0]

            # N-step discount
            target_q = rewards_t + (self.gamma ** self.n_steps) * next_q * (1 - dones_t)

        # TD errors
        td_errors = (current_q - target_q).detach().cpu().numpy()

        # Weighted loss
        loss = (weights_t * F.smooth_l1_loss(current_q, target_q, reduction='none')).mean()

        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), 10.0)
        self.optimizer.step()

        # Update priorities
        if self.use_per:
            self.memory.update_priorities(indices, td_errors)

        self.losses.append(loss.item())

    def update_target(self):
        self.target_net.load_state_dict(self.policy_net.state_dict())


print("EnhancedDQNAgent (Rainbow-lite) defined.")

In [ ]:
def train_agent(agent_config, n_episodes=300, max_steps=8, label=""):
    """Train a DQN variant."""
    env = ImageEnvTricks(image_size=48, max_steps=max_steps)
    agent = EnhancedDQNAgent(n_actions=env.n_actions, image_size=48, **agent_config)

    rewards = []
    psnrs = []

    for ep in range(n_episodes):
        state = env.reset()
        ep_reward = 0

        for step in range(max_steps):
            action = agent.select_action(state)
            next_state, reward, done, info = env.step(action)
            agent.store_transition(state, action, reward, next_state, done)
            agent.optimize()
            state = next_state
            ep_reward += reward
            if done:
                break

        if ep % agent.target_update == 0:
            agent.update_target()

        rewards.append(ep_reward)
        psnrs.append(info['psnr'])

        if (ep + 1) % 100 == 0:
            print(f"  [{label}] Ep {ep+1}: Avg Reward={np.mean(rewards[-50:]):.3f}, PSNR={np.mean(psnrs[-50:]):.2f}")

    return rewards, psnrs, agent


# Train different configurations
print("Training Vanilla DQN...")
r_vanilla, p_vanilla, _ = train_agent(
    {'use_double': False, 'use_per': False, 'use_dueling': False, 'n_steps': 1},
    n_episodes=300, label="Vanilla"
)

print("\nTraining Double + Dueling + PER + 3-step...")
r_enhanced, p_enhanced, agent_enhanced = train_agent(
    {'use_double': True, 'use_per': True, 'use_dueling': True, 'n_steps': 3},
    n_episodes=300, label="Enhanced"
)

In [ ]:
# Before/After Comparison

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
window = 25

# Rewards comparison
ax = axes[0]
ax.plot(r_vanilla, alpha=0.2, color='red')
ax.plot(r_enhanced, alpha=0.2, color='blue')
if len(r_vanilla) >= window:
    sm_v = np.convolve(r_vanilla, np.ones(window)/window, mode='valid')
    sm_e = np.convolve(r_enhanced, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(r_vanilla)), sm_v, color='red', linewidth=2.5, label='Vanilla DQN')
    ax.plot(range(window-1, len(r_enhanced)), sm_e, color='blue', linewidth=2.5, label='Double+Dueling+PER+3step')
ax.set_xlabel('Episode', fontsize=11)
ax.set_ylabel('Episode Reward', fontsize=11)
ax.set_title('Reward: Vanilla vs Enhanced DQN', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# PSNR comparison
ax = axes[1]
if len(p_vanilla) >= window:
    sm_pv = np.convolve(p_vanilla, np.ones(window)/window, mode='valid')
    sm_pe = np.convolve(p_enhanced, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(p_vanilla)), sm_pv, color='red', linewidth=2.5, label='Vanilla DQN')
    ax.plot(range(window-1, len(p_enhanced)), sm_pe, color='blue', linewidth=2.5, label='Enhanced DQN')
ax.set_xlabel('Episode', fontsize=11)
ax.set_ylabel('PSNR (dB)', fontsize=11)
ax.set_title('Image Quality: Vanilla vs Enhanced', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle('Effect of DQN Tricks on Image Processing Performance', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Final stats
print(f"\n{'='*60}")
print(f"Final Performance (last 50 episodes):")
print(f"  Vanilla DQN:  Reward={np.mean(r_vanilla[-50:]):.3f}, PSNR={np.mean(p_vanilla[-50:]):.2f} dB")
print(f"  Enhanced DQN: Reward={np.mean(r_enhanced[-50:]):.3f}, PSNR={np.mean(p_enhanced[-50:]):.2f} dB")
print(f"{'='*60}")

---

## 7. Summary

### Tricks Comparison Table

| Trick | Problem Solved | Key Formula | Improvement |
|-------|---------------|-------------|-------------|
| **PER** | Uniform sampling wastes capacity | $P(i) = \frac{p_i^\alpha}{\sum_k p_k^\alpha}$ | More efficient use of experience |
| **N-step** | 1-step has high bias | $G_t^{(n)} = \sum_{k=0}^{n-1} \gamma^k r_{t+k} + \gamma^n V(s_{t+n})$ | Better credit assignment |
| **Double DQN** | Overestimation bias | $y = r + \gamma Q(s', \arg\max_{a'} Q(s', a'; \theta); \theta^-)$ | Unbiased value estimates |
| **Dueling DQN** | Not all states need action discrimination | $Q = V(s) + A(s,a) - \bar{A}$ | Better state value learning |
| **Noisy Nets** | $\epsilon$-greedy is state-independent | $w = \mu + \sigma \odot \varepsilon$ | Adaptive, state-dependent exploration |

### Rainbow DQN

Rainbow (Hessel et al., 2018) combines **all** these tricks:

$$\text{Rainbow} = \text{DQN} + \text{Double} + \text{Dueling} + \text{PER} + \text{N-step} + \text{NoisyNet} + \text{Distributional}$$

Each component provides complementary improvements, and their combination achieves state-of-the-art on Atari benchmarks.

### For Image Processing Tasks

- **PER** is critical — image transformations have varying importance
- **Dueling** helps — many image states have similar values (only specific degradations need specific actions)
- **Double DQN** — always use it (free improvement, no extra cost)
- **N-step** ($n=3$) — captures longer-term effects of image operations